# Climate Risk, Value, and Investing — Exam Capstone Pipeline
**Student Roll Number:** `msd25006` | **Academic Year:** 2026 | **Version:** V003
**Assigned Universe:** Southeast Asia (10 Holdings) | Sectors: Utilities & Consumer Staples
**Focal Physical Asset:** Urban Data Centre (Singapore/KL Corridor) — Hazard: Extreme Heat
**Focal Labelled Bond:** Corporate Sustainability-Linked Bond (SLB)

> This notebook reproduces the full quantitative pipeline across all six course blocks (A–F) in under 3 minutes.

## Stage 0: Universe Initialization (10 Southeast Asian Holdings)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RNG = np.random.default_rng(42)
pd.set_option('display.width', 140)

tickers = ['SEMB', 'TNB', 'MER', 'GULF', 'PGAS', 'CPALL', 'WILM', 'THBEV', 'URC', 'INDF']
company_names = [
    'Sembcorp Industries Ltd', 'Tenaga Nasional Berhad', 'Manila Electric Company (Meralco)',
    'Gulf Energy Development PCL', 'PT Perusahaan Gas Negara Tbk', 'CP ALL PCL',
    'Wilmar International Ltd', 'Thai Beverage PCL', 'Universal Robina Corporation',
    'PT Indofood CBP Sukses Makmur Tbk'
]
countries = ['Singapore', 'Malaysia', 'Philippines', 'Thailand', 'Indonesia',
             'Thailand', 'Singapore', 'Thailand', 'Philippines', 'Indonesia']
sectors = ['Utilities', 'Utilities', 'Utilities', 'Utilities', 'Utilities',
           'Consumer Staples', 'Consumer Staples', 'Consumer Staples', 'Consumer Staples', 'Consumer Staples']
revenues_musd = [5820.0, 11450.0, 7840.0, 3120.0, 3650.0, 24800.0, 67100.0, 7950.0, 2850.0, 4420.0]
market_caps_musd = [7100.0, 16200.0, 8900.0, 14500.0, 2400.0, 15800.0, 14900.0, 9800.0, 4100.0, 7200.0]
scope1_kt = [2200.0, 6800.0, 1650.0, 1150.0, 850.0, 180.0, 4800.0, 320.0, 110.0, 160.0]
scope2_kt = [590.0, 300.0, 190.0, 95.0, 60.0, 2180.0, 16020.0, 395.0, 145.0, 210.0]

holdings_df = pd.DataFrame({
    'ticker': tickers,
    'company_name': company_names,
    'country': countries,
    'sector': sectors,
    'revenue_musd': revenues_musd,
    'market_cap_musd': market_caps_musd,
    'benchmark_weight': np.repeat(0.10, 10),
    'scope1_emissions_kt': scope1_kt,
    'scope2_emissions_kt': scope2_kt,
})
holdings_df['emis_intensity_tco2_per_musd'] = (
    (holdings_df['scope1_emissions_kt'] + holdings_df['scope2_emissions_kt']) * 1000.0 / holdings_df['revenue_musd']
).round(1)
holdings_df[['ticker', 'country', 'sector', 'revenue_musd', 'emis_intensity_tco2_per_musd']]


## Stage 1: Frame Decision & Financial Transmission (Block A)

In [2]:
frame_df = holdings_df.copy()
frame_df['physical_exposure_score'] = [0.42, 0.48, 0.72, 0.40, 0.38, 0.68, 0.75, 0.62, 0.65, 0.70]
frame_df['transition_exposure_score'] = [0.78, 0.88, 0.65, 0.70, 0.60, 0.28, 0.42, 0.32, 0.25, 0.30]
frame_df['primary_channel'] = np.where(
    frame_df['transition_exposure_score'] >= frame_df['physical_exposure_score'], 'transition', 'physical'
)
frame_df[['ticker', 'sector', 'physical_exposure_score', 'transition_exposure_score', 'primary_channel']]


## Stage 2: Climate Data, Quality Report & WACI (Block B)

In [3]:
data_df = frame_df.copy()
data_df['hazard_heat_index'] = [0.65, 0.70, 0.78, 0.72, 0.75, 0.80, 0.85, 0.76, 0.74, 0.82]
data_df.loc[4, 'emis_intensity_tco2_per_musd'] = np.nan  # Realistic disclosure gap

coverage_ratio = float(data_df['emis_intensity_tco2_per_musd'].notna().mean())
sector_median = data_df[data_df['sector'] == 'Utilities']['emis_intensity_tco2_per_musd'].median()
data_df['emis_intensity_imputed'] = data_df['emis_intensity_tco2_per_musd'].fillna(sector_median)

waci_reported = float(np.nansum(data_df['benchmark_weight'] * data_df['emis_intensity_tco2_per_musd']))
waci_full = float(np.sum(data_df['benchmark_weight'] * data_df['emis_intensity_imputed']))
print(f'Data Coverage Ratio: {coverage_ratio:.1%}')
print(f'Reported Baseline WACI: {waci_reported:.1f} tCO2e/MUSD')
print(f'Imputed Portfolio WACI: {waci_full:.1f} tCO2e/MUSD')


## Stage 3: Climate-Adjusted Valuation under Dual NGFS Scenarios (Blocks A-B)

In [4]:
h = data_df[data_df['ticker'] == 'SEMB'].iloc[0]
H = 10; years = np.arange(1, H + 1)
annual_emis_kt = (h['scope1_emissions_kt'] + h['scope2_emissions_kt'])

def dcf_val(carbon_price_path, pass_through=0.50, wacc=0.082, g=0.02, tax=0.17):
    rev = h['revenue_musd'] * (1.03) ** years
    ebit = rev * 0.165
    carbon_cost = (annual_emis_kt * 1000.0 * carbon_price_path / 1e6) * (1.0 - pass_through)
    capex = rev * 0.055
    fcf = ebit * (1.0 - tax) - capex - carbon_cost
    disc = (1.0 + wacc) ** years
    return float((fcf / disc).sum() + (fcf[-1] * (1.0 + g) / (wacc - g)) / disc[-1])

ev_base = dcf_val(np.repeat(5.0, H))
ev_scen1 = dcf_val(np.linspace(25, 40, H))
ev_scen2 = dcf_val(np.linspace(25, 140, H))
print(f'Baseline EV ($5/t): ${ev_base:.1f}M')
print(f'Current Policies EV ($25-40/t): ${ev_scen1:.1f}M ({((ev_scen1/ev_base)-1)*100:.1f}%)')
print(f'Net Zero 2050 EV ($25-140/t): ${ev_scen2:.1f}M ({((ev_scen2/ev_base)-1)*100:.1f}%)')


## Stage 4: ESG Ratings Reconciliation & Stewardship (Block C)

In [5]:
esg_df = data_df[['ticker', 'company_name', 'sector']].copy()
esg_df['score_A_disclosure'] = [9, 5, 6, 6, 4, 7, 8, 7, 5, 5]
esg_df['score_B_alignment'] = [8, 3, 4, 5, 4, 6, 4, 6, 5, 4]
esg_df['ratings_divergence'] = (esg_df['score_A_disclosure'] - esg_df['score_B_alignment']).abs()
print(f'Mean Ratings Divergence: {esg_df["ratings_divergence"].mean():.2f}')
esg_df


## Stage 5: Physical Risk Transfer on Urban Data Centre (Block D)

In [6]:
ASSET_VALUE = 85.0; N_SIMS = 25000
p_heat = 0.18; sev_m = 0.25; sev_s = 0.12
raw_loss = np.clip(RNG.normal(sev_m, sev_s, N_SIMS), 0, 1) * (RNG.random(N_SIMS) < p_heat) * ASSET_VALUE

ADAPT_EFF = 0.45; ATTACH = 3.0; LIMIT = 20.0
loss_adapt = raw_loss * (1.0 - ADAPT_EFF)
retained_ins = raw_loss - np.clip(raw_loss - ATTACH, 0, LIMIT)
comb_loss = loss_adapt - np.clip(loss_adapt - ATTACH, 0, LIMIT)

risk_summary = pd.DataFrame([
    {'strategy': 'No Action', 'EAL': round(float(raw_loss.mean()), 2), 'P95': round(float(np.percentile(raw_loss, 95)), 2)},
    {'strategy': 'Adaptation', 'EAL': round(float(loss_adapt.mean()), 2), 'P95': round(float(np.percentile(loss_adapt, 95)), 2)},
    {'strategy': 'Insurance', 'EAL': round(float(retained_ins.mean()), 2), 'P95': round(float(np.percentile(retained_ins, 95)), 2)},
    {'strategy': 'Combined', 'EAL': round(float(comb_loss.mean()), 2), 'P95': round(float(np.percentile(comb_loss, 95)), 2)}
])
risk_summary


## Stage 6: Corporate Sustainability-Linked Bond Pricing (Block E)

In [7]:
FACE = 100.0; BASE_COUPON = 0.0475; PENALTY_STEP = 0.0025; MATURITY = 5; YLD = 0.0440
def slb_price(target_met):
    c = BASE_COUPON if target_met else (BASE_COUPON + PENALTY_STEP)
    cfs = [FACE * c] * (MATURITY - 1) + [FACE * (1.0 + c)]
    return sum(cf / ((1.0 + YLD) ** (t + 1)) for t, cf in enumerate(cfs))

p_met = round(slb_price(True), 2)
p_miss = round(slb_price(False), 2)
print(f'SLB Target Met Price: ${p_met} | Target Missed (+25 bps): ${p_miss}')


## Stage 7: Integration & Capital Allocation Decision (Block F)

In [8]:
signal_df = data_df[['ticker', 'company_name', 'sector', 'benchmark_weight']].copy()
max_emis = data_df['emis_intensity_imputed'].max()
signal_df['carbon_penalty'] = (data_df['emis_intensity_imputed'] / max_emis).round(3)
signal_df['esg_factor'] = (esg_df['score_A_disclosure'] / 10.0).round(3)
signal_df['physical_vulnerability'] = (data_df['hazard_heat_index'] * frame_df['physical_exposure_score']).round(3)

signal_df['composite_score'] = (
    0.40 * signal_df['esg_factor'] - 0.30 * signal_df['carbon_penalty'] - 0.30 * signal_df['physical_vulnerability']
).round(3)

cond = [signal_df['composite_score'] >= 0.150, signal_df['composite_score'] <= -0.050]
recs = ['Overweight (+3.0%)', 'Underweight (-3.0%)']
signal_df['recommendation'] = np.select(cond, recs, default='Neutral (Hold Benchmark)')
signal_df.sort_values('composite_score', ascending=False)[['ticker', 'company_name', 'composite_score', 'recommendation']]


## Acceptance Checks & Assertions

In [9]:
assert len(holdings_df) == 10, 'Expected 10 holdings'
assert frame_df['primary_channel'].isin(['physical', 'transition']).all()
assert 0 <= coverage_ratio <= 1 and waci_full > 0
assert ev_base > ev_scen2
assert risk_summary.set_index('strategy').loc['Adaptation', 'EAL'] < risk_summary.set_index('strategy').loc['No Action', 'EAL']
assert p_miss > p_met
print('ALL ACCEPTANCE CHECKS PASSED SUCCESSFULLY — ALL SIX BLOCKS PRODUCED EVIDENCE!')
